# 01 — Line Segment Intersection

Two paths cross if any of their segments cross. Before we can ask "does this missile path enter that country?", we need a reliable answer to a simpler question: **do these two line segments intersect?**

This notebook builds that test from scratch using the **orientation method** — a clean approach based on which side of a line a point falls on, requiring only arithmetic.

## Setup

In [ ]:
import json
from pathlib import Path
from ipyleaflet import Map, GeoJSON, basemaps

DATA_DIR = Path("data")

with open(DATA_DIR / "paths.geojson") as f:
    module_paths = json.load(f)

print(f"{len(module_paths['features'])} paths loaded")
for feat in module_paths["features"]:
    p = feat["properties"]
    print(f"  {p['name']:8}  {p['origin']} → {p['destination']}")

## What Does Intersection Mean?

Two line segments intersect if they share at least one point.

```
Intersecting:          Not intersecting (parallel):    Not intersecting (short):

   A ────X──── B          A ──────── B                  A ──── B
         │                C ──────── D                             C ──── D
   C ────╯
         D
```

The third case is the tricky one — the two lines *would* meet if extended, but the segments are not long enough to actually touch. A correct test must handle this. The key insight: **two segments intersect if and only if each segment straddles the line containing the other.**

## Visualizing Intersecting vs Non-Intersecting Pairs

Before writing any math, let's look at a few pairs on a map. Three cases:

- **Red pair** — segments clearly cross
- **Blue pair** — segments are parallel, no cross
- **Green pair** — segments aim at the same point but are too short to meet

In [ ]:
def make_segment(p1, p2, color, name=""):
    return {
        "type": "Feature",
        "properties": {"name": name, "color": color},
        "geometry": {"type": "LineString", "coordinates": [p1, p2]}
    }

pairs = {
    "type": "FeatureCollection",
    "features": [
        # Red — crossing
        make_segment([-99.0, 34.0], [-97.0, 32.0], "#e74c3c", "cross-A"),
        make_segment([-99.0, 32.0], [-97.0, 34.0], "#e74c3c", "cross-B"),
        # Blue — parallel, no cross
        make_segment([-96.0, 34.5], [-94.0, 34.5], "#2980b9", "parallel-A"),
        make_segment([-96.0, 33.5], [-94.0, 33.5], "#2980b9", "parallel-B"),
        # Green — would cross if extended, but too short
        make_segment([-93.0, 34.0], [-92.0, 33.5], "#27ae60", "short-A"),
        make_segment([-91.5, 34.0], [-90.5, 33.5], "#27ae60", "short-B"),
    ]
}

m = Map(center=(33.5, -96), zoom=6, basemap=basemaps.CartoDB.Positron)
m.add(GeoJSON(
    data=pairs,
    style_callback=lambda f: {"color": f["properties"]["color"], "weight": 3}
))
m

## Orientation: Which Side of a Line?

Given three points P, Q, R — imagine standing at P, facing Q. Is R to your **left** (counterclockwise), to your **right** (clockwise), or directly ahead (collinear)?

```
        R                    R
       ↗                      ↘
P ──→ Q                 P ──→ Q
   CCW (left)               CW (right)
```

This is computed with a **cross product**:

```
val = (Q.x − P.x) × (R.y − Q.y) − (Q.y − P.y) × (R.x − Q.x)

val > 0  →  counterclockwise
val < 0  →  clockwise
val = 0  →  collinear
```

No trigonometry, no square roots — just multiplication and subtraction.

In [ ]:
def orientation(p, q, r):
    """
    Returns the orientation of the ordered triplet (p, q, r).
    Points are [lon, lat] pairs.

      > 0  counterclockwise
      < 0  clockwise
      = 0  collinear
    """
    return (q[0] - p[0]) * (r[1] - q[1]) - (q[1] - p[1]) * (r[0] - q[0])


# Three concrete examples
P = [0, 0]
Q = [1, 0]

R_left  = [1, 1]   # R is above PQ → CCW
R_right = [1, -1]  # R is below PQ → CW
R_col   = [2, 0]   # R is on PQ extended → collinear

print(f"R left  (CCW):    {orientation(P, Q, R_left):+.1f}")
print(f"R right (CW):     {orientation(P, Q, R_right):+.1f}")
print(f"R collinear:      {orientation(P, Q, R_col):+.1f}")

## The Intersection Test

Segment **AB** and segment **CD** intersect when:

1. **C and D are on opposite sides of line AB**  
   `orientation(A, B, C)` and `orientation(A, B, D)` have opposite signs

2. **A and B are on opposite sides of line CD**  
   `orientation(C, D, A)` and `orientation(C, D, B)` have opposite signs

Both conditions must hold. Condition 1 alone means C and D straddle the *infinite line* through AB — not necessarily the segment. Condition 2 eliminates the "too short" case.

```
       C                    C
       │                    │
A ─────X───── B    A ─── B  │     ← condition 1 holds, condition 2 fails
       │                    │
       D                    D
    INTERSECT           NO INTERSECT
```

In [ ]:
def on_segment(p, q, r):
    """True if collinear point q lies on segment p-r."""
    return (min(p[0], r[0]) <= q[0] <= max(p[0], r[0]) and
            min(p[1], r[1]) <= q[1] <= max(p[1], r[1]))


def segments_intersect(p1, p2, p3, p4):
    """
    Returns True if segment p1-p2 intersects segment p3-p4.
    Points are [lon, lat] pairs.
    Handles collinear / touching-endpoint edge cases.
    """
    d1 = orientation(p3, p4, p1)
    d2 = orientation(p3, p4, p2)
    d3 = orientation(p1, p2, p3)
    d4 = orientation(p1, p2, p4)

    # General case
    if (d1 > 0 and d2 < 0 or d1 < 0 and d2 > 0) and \
       (d3 > 0 and d4 < 0 or d3 < 0 and d4 > 0):
        return True

    # Collinear cases — point lies exactly on the other segment
    if d1 == 0 and on_segment(p3, p1, p4): return True
    if d2 == 0 and on_segment(p3, p2, p4): return True
    if d3 == 0 and on_segment(p1, p3, p2): return True
    if d4 == 0 and on_segment(p1, p4, p2): return True

    return False


# Quick sanity checks
A, B = [0, 0], [2, 2]
C, D = [0, 2], [2, 0]
print("X-cross:     ", segments_intersect(A, B, C, D))   # True

E, F = [3, 0], [5, 0]
G, H = [3, 1], [5, 1]
print("Parallel:    ", segments_intersect(E, F, G, H))   # False

I, J = [0, 0], [1, 0]
K, L = [2, 0], [3, 0]
print("Collinear gap:", segments_intersect(I, J, K, L))  # False

## Edge Cases

Three situations require special handling:

| Case | Description | `orientation` value | Result |
|---|---|---|---|
| **Touching endpoints** | One segment ends exactly on the other | 0 | Intersects |
| **T-junction** | Endpoint of one lies in interior of other | 0 | Intersects |
| **Collinear overlap** | Both segments on the same infinite line, overlapping | 0, 0 | Intersects |
| **Collinear gap** | Both on same line, not touching | 0, 0 | Does NOT intersect |

The `on_segment` helper resolves these — when orientation is zero (collinear), it checks whether the point actually falls within the segment's bounding box.

In [ ]:
# Touching endpoint: B lies exactly on CD
A, B = [0, 0], [1, 1]
C, D = [0, 2], [2, 0]   # B=[1,1] is on CD
print("Touching endpoint:  ", segments_intersect(A, B, C, D))   # True

# T-junction: C lies in the interior of AB
A, B = [0, 0], [4, 0]
C, D = [2, 0], [2, 2]   # C=[2,0] is on AB
print("T-junction:         ", segments_intersect(A, B, C, D))   # True

# Collinear overlap
A, B = [0, 0], [3, 0]
C, D = [2, 0], [5, 0]
print("Collinear overlap:  ", segments_intersect(A, B, C, D))   # True

# Collinear gap — same line, no overlap
A, B = [0, 0], [1, 0]
C, D = [2, 0], [3, 0]
print("Collinear gap:      ", segments_intersect(A, B, C, D))   # False

## Applying the Test to Real Path Segments

Each path in `paths.geojson` is a chain of segments. We can test any two paths against each other by iterating over their segments.

Here we extract the single segment from each path (2-point paths → 1 segment each) and check every pair.

In [ ]:
# Extract segments from each path
def path_segments(feature):
    """Return list of (p1, p2) tuples for every consecutive pair in the LineString."""
    coords = feature["geometry"]["coordinates"]
    return [(coords[i], coords[i + 1]) for i in range(len(coords) - 1)]


paths = module_paths["features"]

print("Path-vs-path intersection check:")
for i in range(len(paths)):
    for j in range(i + 1, len(paths)):
        name_i = paths[i]["properties"]["name"]
        name_j = paths[j]["properties"]["name"]
        segs_i = path_segments(paths[i])
        segs_j = path_segments(paths[j])
        hit = any(
            segments_intersect(a, b, c, d)
            for a, b in segs_i
            for c, d in segs_j
        )
        print(f"  {name_i} × {name_j}: {'INTERSECT' if hit else 'no cross'}")

## Exercise A

Inspect the four segment pairs below **by eye** first — predict whether each pair intersects or not, then run `segments_intersect` to check your prediction.

```python
pairs = [
    # pair 1
    ([0, 0], [4, 4],  [0, 4], [4, 0]),
    # pair 2
    ([0, 0], [2, 0],  [3, 0], [5, 0]),
    # pair 3
    ([0, 0], [3, 0],  [2, 0], [2, 3]),
    # pair 4
    ([0, 2], [4, 2],  [5, 0], [5, 4]),
]
```

Print your prediction and the function result side by side for each pair.

In [ ]:
pairs = [
    ([0, 0], [4, 4],  [0, 4], [4, 0]),   # pair 1
    ([0, 0], [2, 0],  [3, 0], [5, 0]),   # pair 2
    ([0, 0], [3, 0],  [2, 0], [2, 3]),   # pair 3
    ([0, 2], [4, 2],  [5, 0], [5, 4]),   # pair 4
]

# For each pair, write your prediction as True/False, then call segments_intersect
my_predictions = [True, False, True, False]   # replace with your guesses

# Your code here: compare predictions to segments_intersect results

## Exercise B

Implement `segments_intersect` yourself using `orientation` and `on_segment` as building blocks. Both helpers are provided — your job is the logic that combines them.

Verify your implementation against the four test cases at the bottom of the cell.

In [ ]:
def orientation(p, q, r):
    return (q[0] - p[0]) * (r[1] - q[1]) - (q[1] - p[1]) * (r[0] - q[0])

def on_segment(p, q, r):
    return (min(p[0], r[0]) <= q[0] <= max(p[0], r[0]) and
            min(p[1], r[1]) <= q[1] <= max(p[1], r[1]))

def segments_intersect(p1, p2, p3, p4):
    """
    Returns True if segment p1-p2 intersects segment p3-p4.
    Hint: compute 4 orientation values, check general case first,
    then handle collinear edge cases with on_segment.
    """
    # Your code here
    pass


# Verification — all four should pass
assert segments_intersect([0,0],[4,4],[0,4],[4,0]) == True,  "X-cross should intersect"
assert segments_intersect([0,0],[2,0],[3,0],[5,0]) == False, "Collinear gap should not intersect"
assert segments_intersect([0,0],[3,0],[2,0],[2,3]) == True,  "T-junction should intersect"
assert segments_intersect([0,2],[4,2],[5,0],[5,4]) == False, "Parallel miss should not intersect"
print("All assertions passed.")

---

## Check Your Understanding

**1.** When do two segments NOT intersect even though their containing lines would meet if extended?

**2.** Why do endpoint touches count as intersections, and why does the `on_segment` check matter for the collinear case?

```python
# No code needed — answer in your own words
```

## Next

In [02 — Line vs. Polygon Basics](./02-Line_vs_Polygon_Basics.ipynb), we extend the segment test to polygons — looping over every edge of a polygon to ask whether a path crosses any of them.